# Clinical trials corpus — load and publish

Loads `data/clinical_trials.parquet` (597,691 trials, 49 columns, 2.23 GiB on disk) as a
Hugging Face `Dataset` and pushes it to an existing dataset repo on the Hub.

**Before running anything:**

1. Install `datasets`. It is not in this venv, and neither is `pandas` any more — the
   outputs saved in the previous version of this notebook came from an earlier run, and
   its `import pandas` would fail today. `datasets` pulls `pandas` back in as a
   dependency. The corporate TLS proxy means `uv` needs `--native-tls`, or it dies with
   `invalid peer certificate: UnknownIssuer`:

   ```bash
   uv add --native-tls datasets
   ```

2. Export your token in the shell **before** starting the kernel:

   ```bash
   export HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxx
   ```

   Putting it in `.env` is not reliable here — `load_dotenv` will not override a
   variable that is already set in the real environment.

3. Set `REPO_ID` in the config cell below, or export `HF_DATASET_REPO`.


# Clinical Trials 600k Data Uplaod


In [ ]:
import ssl
import sys

if sys.version_info >= (3, 14) and "truststore" in getattr(
    ssl.SSLContext, "__module__", ""
):
    for _name in ("pip._vendor.truststore", "truststore"):
        try:
            __import__(_name, fromlist=["extract_from_ssl"]).extract_from_ssl()
        except Exception:  # not that copy of truststore; try the next
            continue
        if "truststore" not in ssl.SSLContext.__module__:
            print(f"removed truststore SSL injection ({_name})")
            break

print("python         :", sys.version.split()[0])
print("ssl.SSLContext :", ssl.SSLContext.__module__)

In [ ]:
import os
from pathlib import Path

PARQUET = Path("../data/clinical_trials.parquet").resolve()

PLACEHOLDER = "YOUR-USER/YOUR-DATASET"
REPO_ID = os.environ.get("HF_DATASET_REPO", PLACEHOLDER)
SPLIT = "train"
MAX_SHARD_SIZE = "500MB"

CACHE_DIR = os.environ.get("HF_DATASETS_CACHE") or None

assert PARQUET.exists(), f"corpus not found: {PARQUET}"
print(f"parquet   : {PARQUET}")
print(f"size      : {PARQUET.stat().st_size / 1024 ** 3:.2f} GiB")
print(f"repo      : {REPO_ID}")
print(f"token set : {bool(os.environ.get('HF_TOKEN'))}")

In [ ]:
from datasets import Dataset

ds = Dataset.from_parquet(str(PARQUET), cache_dir=CACHE_DIR, split=SPLIT)
ds

In [ ]:
print(f"rows       : {ds.num_rows:,}")
print(f"columns    : {len(ds.column_names)}")
print(
    f"arrow bytes: {ds.data.nbytes / 1024 ** 3:.2f} GiB (memory-mapped, not resident)"
)
print()
for name, feature in ds.features.items():
    print(f"  {name:<28} {str(feature)[:90]}")

In [ ]:
import json

# Indexing one row decodes only that row, so this stays cheap on 597k rows.
for key, value in ds[12].items():
    text = json.dumps(value, default=str, ensure_ascii=False)
    print(f"{key:<28} {text[:150]}{'...' if len(text) > 150 else ''}")

In [ ]:
assert (
    REPO_ID != PLACEHOLDER
), "Set REPO_ID in the config cell, or export HF_DATASET_REPO, before pushing."
assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN is not set. Export it in the shell and restart the kernel -- otherwise this "
    "fails with a 401 having uploaded nothing."
)

ds.push_to_hub(
    REPO_ID,
    split=SPLIT,
    max_shard_size=MAX_SHARD_SIZE,
    commit_message=f"Add ClinicalTrials.gov corpus ({ds.num_rows:,} trials)",
)

In [ ]:
from datasets import load_dataset

remote = load_dataset(REPO_ID, split=SPLIT, streaming=True)
first = next(iter(remote))

print(f"columns match local : {sorted(first) == sorted(ds.column_names)}")
print(f"first nct_id        : {first.get('nct_id')}")
print(f"https://huggingface.co/datasets/{REPO_ID}")

# Decision Model Fine-tuning Data Upload


In [1]:
import os
from pathlib import Path

MAX_SHARD_SIZE = "500MB"
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
SYNTHETIC_DATA_DIR = DATA_DIR / "synthetic"
CACHE_DIR = os.environ.get("HF_DATASETS_CACHE") or None

In [2]:
from datasets import Dataset

train_ds = Dataset.from_json(
    str(SYNTHETIC_DATA_DIR / "intake_v1.train.jsonl"),
    cache_dir=CACHE_DIR,
    split="train",
)
eval_ds = Dataset.from_json(
    str(SYNTHETIC_DATA_DIR / "intake_v1.eval.jsonl"),
    cache_dir=CACHE_DIR,
    split="eval",
)
manifest_ds = Dataset.from_json(
    str(SYNTHETIC_DATA_DIR / "intake_v1.manifest.jsonl"),
    cache_dir=CACHE_DIR,
    split="manifest",
)

/workspaces/trial-pilot-clinical-trials/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 8835 examples [00:00, 9742.69 examples/s]
Generating eval split: 1050 examples [00:00, 12620.23 examples/s]
Generating manifest split: 9885 examples [00:00, 24344.84 examples/s]


In [3]:
import json

for key, value in train_ds[12].items():
    text = json.dumps(value, default=str, ensure_ascii=False)
    print(f"{key:<28} {text[:150]}{'...' if len(text) > 150 else ''}")

state                        "Hi, could you point me toward the most up-to-date guidelines for breast cancer screening? Just looking for a general summary, not asking about a spec...
questions                    {"sex": {"type": "choice", "instructions": "What is the biological sex of the patient described?", "criteria": {"female": "the patient is female", "ma...
expected                     {"sex": "unknown", "age_band": "unknown", "smoking_status": "unknown", "pregnancy_menopausal_status": "unknown", "performance_status": "unknown", "sex...
seed                         15
generator                    "gemini-3.5-flash-lite"
domain                       "off_topic"
presentation                 "clinician"
gold                         null


In [ ]:
train_ds.push_to_hub(
    "DenCT/laya-decisions-intake-10k",
    split="train",
    token="",
    max_shard_size=MAX_SHARD_SIZE,
    commit_message=f"Add ClinicalTrials.gov corpus ({train_ds.num_rows:,} trials)",
)
eval_ds.push_to_hub(
    "DenCT/laya-decisions-intake-10k",
    split="validation",
    token="",
    max_shard_size=MAX_SHARD_SIZE,
    commit_message=f"Add ClinicalTrials.gov corpus ({eval_ds.num_rows:,} trials)",
)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 11.83ba/s]
Processing Files (1 / 1): 100%|██████████| 1.95MB / 1.95MB,  186kB/s  







Processing Files (1 / 1): 100%|██████████| 1.95MB / 1.95MB,  167kB/s  
New Data Upload: 100%|██████████|  202kB /  202kB, 17.8kB/s  
Validating: ██████████| 100%            
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.34s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 208.39ba/s]
Processing Files (1 / 1): 100%|██████████|  227kB /  227kB, 21.7kB/s  







Processing Files (1 / 1): 100%|██████████|  227kB /  227kB, 19.3kB/s  
New Data Upload: 100%|██████████| 64.8kB / 64.8kB, 5.67kB/s  
Validating: ██████████| 100%            
Uploading the dataset

CommitInfo(commit_url='https://huggingface.co/datasets/DenCT/laya-decisions-intake-10k/commit/dcee15e6ac718319a0f989b165bc87b30013c7c0', commit_message='Add ClinicalTrials.gov corpus (1,050 trials)', commit_description='', oid='dcee15e6ac718319a0f989b165bc87b30013c7c0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/DenCT/laya-decisions-intake-10k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='DenCT/laya-decisions-intake-10k'), pr_revision=None, pr_num=None)